## Load trained sparse EBMs and data

In [ ]:
import joblib
import copy

folds_ebm = joblib.load("models/folds_ebm.joblib")

fold_0 = folds_ebm[0]
fold_0 = copy.deepcopy(fold_0)

## Add age group column to holdout set, get group fractions and incidences

In [ ]:
from scripts.ebm_fairness import add_age_group_fold, get_incidence_rate_fold

fold_0, group_fractions = add_age_group_fold(fold_0)

print(group_fractions)

incidence_rate = get_incidence_rate_fold(fold_0)

print(incidence_rate)

## Add group intersections column to holdout set, get group fractions and incidences

In [ ]:
from scripts.ebm_fairness import add_groups_fold, get_incidence_rate_intersections_fold

fold_0_g, group_fractions_g = add_groups_fold(fold_0)

print(group_fractions_g)

group_fractions_db = group_fractions_g.to_frame(name='intersect_counts').reset_index()
group_fractions_db.columns = ['groups', 'count']  # Explicitly name the columns

print(group_fractions_db)

# Save to CSV
group_fractions_db.to_csv('results/group_intersections_counts.csv', index=False)

incidence_rate_g = get_incidence_rate_intersections_fold(fold_0)

print(incidence_rate_g)

group_pos_incid_db = incidence_rate_g.to_frame(name='intersect_incidences').reset_index()
group_pos_incid_db.columns = ['groups', 'positive_incidences']  # Explicitly name the columns

print(group_pos_incid_db)

# Save to CSV
group_pos_incid_db.to_csv('results/group_intersections_incidences.csv', index=False)

## Plot cumulative gain curve

In [ ]:
from scripts.ebm_fairness import compute_pred_probs, plot_cumulative_gain_curve, population_share_for_recall

fold_0 = compute_pred_probs(fold_0)

plot_cumulative_gain_curve(fold_0, pos_label=1, n_bins=100)

fraction_needed = population_share_for_recall(fold_0, target_recall=0.8)

print(f"You need to target {fraction_needed:.2%} of the population to capture 80% of positives.")

## Add binary risk bin to fold based on above target

In [ ]:
from scripts.ebm_fairness import add_binary_risk_column

fold_0, threshold = add_binary_risk_column(fold_0, quant=0.51)

fold_0_g, threshold_g = add_binary_risk_column(fold_0_g, quant=0.51)

print(threshold)

print(threshold_g)

## Get normalized confusion matrices for each age group

In [ ]:
from scripts.ebm_fairness import get_normalized_confusion_matrices_by_group, plot_confusion_matrices_by_group

conf_matrices_prefair = get_normalized_confusion_matrices_by_group(fold_0, 'y_preds', 'alter_gruppen')
plot_confusion_matrices_by_group(conf_matrices_prefair, 'Pre-Fair')

## Get normalized confusion matrices for each group intersection

In [ ]:
from scripts.ebm_fairness import get_normalized_confusion_matrices_by_group, plot_confusion_matrices_by_group

conf_matrices_prefair_g = get_normalized_confusion_matrices_by_group(fold_0_g, 'y_preds', 'gruppen')
plot_confusion_matrices_by_group(conf_matrices_prefair_g, 'Pre-Fair')

## Apply fairness postprocessing age groups

In [ ]:
from scripts.ebm_fairness import get_postprocess_fit_predict

fold_0 = get_postprocess_fit_predict(fold_0)

conf_matrices_fair = get_normalized_confusion_matrices_by_group(fold_0, 'y_preds_fair', 'alter_gruppen')
plot_confusion_matrices_by_group(conf_matrices_fair, 'Fair')

## Apply fairness post-processing group intersections

In [ ]:
from scripts.ebm_fairness import get_postprocess_fit_predict_g

fold_0_g = get_postprocess_fit_predict_g(fold_0)

conf_matrices_fair_g = get_normalized_confusion_matrices_by_group(fold_0_g, 'y_preds_fair', 'gruppen')
plot_confusion_matrices_by_group(conf_matrices_fair_g, 'Fair')

## Get Accuracy and Coverage of both predictors w.r.t age

In [ ]:
from scripts.ebm_fairness import plot_tpr_positive_accuracy

y_true = fold_0['holdout_Y'].squeeze().to_numpy()

plot_tpr_positive_accuracy(y_true, fold_0['y_preds'], 'Age, pre-Fair')

plot_tpr_positive_accuracy(y_true, fold_0['y_preds_fair'], 'Age, Fair')


## Get Accuracy and Coverage of both predictors w.r.t. group intersections

In [ ]:
from scripts.ebm_fairness import plot_tpr_positive_accuracy

y_true = fold_0_g['holdout_Y'].squeeze().to_numpy()

plot_tpr_positive_accuracy(y_true, fold_0_g['y_preds'], 'Intersections, pre-Fair')

plot_tpr_positive_accuracy(y_true, fold_0_g['y_preds_fair'], 'Intersections, Fair')